# PuLP から linopy へ ― xarray ネイティブな最適化ライブラリの実力

このノートブックでは、Python の数理最適化ライブラリ **linopy** を、多くの方に馴染み深い **PuLP** と比較しながら紹介します。

題材は **ローリングホライゾン生産計画**。実務でよく使うパターンを使って、linopy の特長を体感します。

| 評価軸 | PuLP | linopy |
|--------|------|--------|
| モデル構築速度（update） | 64.6 ms/window | **16.9 ms/window (3.8x)** |
| 初回ビルド | 97.8 ms | **44.1 ms (2.2x)** |
| ピークメモリ | 7.4 MB | **1.0 MB (7.5x)** |
| 合計実行時間（51 window） | 7.2 s | **3.8 s (1.9x)** |

（500 製品・9 日ウィンドウ・HiGHS 1.13.1 で計測）

## 0. インストール

Google Colab などで実行する場合は以下を実行してください。

In [ ]:
# Colab の場合のみ実行（ローカル環境では不要）
# !pip install linopy>=0.7.0 pulp>=3.3.2 "highspy!=1.14.0"

## 1. インポート

In [ ]:
import time
import gc
import tracemalloc

import numpy as np
import xarray as xr
import pulp
import linopy

print(f"linopy : {linopy.__version__}")
print(f"PuLP   : {pulp.__version__}")

## 2. 基本的な使い方：小さな例で比較

まず 2 製品・3 日の小さな生産計画問題で、PuLP と linopy のコードを並べて比較します。

**問題の定式化**

$$
\min \sum_{p,t} c^{\text{prod}}_p x_{p,t} + c^{\text{inv}}_p s_{p,t}
$$

$$
s_{p,t} = s_{p,t-1} + x_{p,t} - d_{p,t} \quad (s_{p,-1} = s^{\text{init}}_p)
$$

$$
0 \le s_{p,t} \le s^{\max}_p, \quad x_{p,t} \ge 0
$$

In [ ]:
# 共通データ（小さな例）
NP, NT = 2, 3
demand_s  = np.array([[30, 40, 35], [25, 45, 30]], dtype=float)
max_inv_s = np.array([200, 150], dtype=float)
init_inv_s = np.array([60, 45], dtype=float)
prod_cost_s = np.array([2.0, 3.0])
inv_cost_s  = np.array([0.2, 0.3])

### PuLP での書き方

In [ ]:
prob = pulp.LpProblem("production", pulp.LpMinimize)

# 変数定義：辞書 + ループ
x = {(p, t): pulp.LpVariable(f"x_{p}_{t}", lowBound=0)
     for p in range(NP) for t in range(NT)}
s = {(p, t): pulp.LpVariable(f"s_{p}_{t}", lowBound=0, upBound=max_inv_s[p])
     for p in range(NP) for t in range(NT)}

# 目的関数
prob += pulp.LpAffineExpression(
    [(x[p, t], prod_cost_s[p]) for p in range(NP) for t in range(NT)]
    + [(s[p, t], inv_cost_s[p]) for p in range(NP) for t in range(NT)]
)

# 在庫バランス制約：ループで 1 本ずつ追加
for p in range(NP):
    for t in range(NT):
        prev = init_inv_s[p] if t == 0 else s[p, t - 1]
        prob += s[p, t] == prev + x[p, t] - demand_s[p, t]

prob.solve(pulp.HiGHS(msg=False))
print(f"PuLP  目的関数値: {pulp.value(prob.objective):.2f}")

### linopy での書き方

linopy では変数・制約を **xarray の DataArray** として扱います。  
ループなしで全製品・全日程を一括定義できます。

In [ ]:
coords = {"product": np.arange(NP), "day": np.arange(NT)}

m = linopy.Model()
x_l = m.add_variables(lower=0, coords=coords, name="x")
s_l = m.add_variables(
    lower=0,
    upper=xr.DataArray(max_inv_s, dims=["product"]),
    coords=coords,
    name="s",
)

da_d    = xr.DataArray(demand_s,  dims=["product", "day"])
da_init = xr.DataArray(init_inv_s, dims=["product"])
da_pc   = xr.DataArray(prod_cost_s, dims=["product"])
da_ic   = xr.DataArray(inv_cost_s,  dims=["product"])

# 目的関数：xarray ブロードキャストで一括
m.add_objective((da_pc * x_l + da_ic * s_l).sum())

# day=0 の制約
m.add_constraints(
    s_l.isel(day=0) - x_l.isel(day=0) == da_init - da_d.isel(day=0),
    name="balance_0",
)
# day=1..NT-1 の制約：1 回の add_constraints で全日程を一括追加
inner = np.arange(1, NT)
s_cur  = s_l.isel(day=slice(1, None)).assign_coords(day=inner)
s_prev = s_l.isel(day=slice(0, -1)).assign_coords(day=inner)
x_cur  = x_l.isel(day=slice(1, None)).assign_coords(day=inner)
m.add_constraints(
    s_cur - s_prev - x_cur == -da_d.isel(day=slice(1, None)).assign_coords(day=inner),
    name="balance_inner",
)

m.solve(solver_name="highs", output_flag=False, log_to_console=False)
print(f"linopy 目的関数値: {m.objective.value:.2f}")

## 3. ローリングホライゾン問題

実務では「今後 9 日分の計画を立て、7 日実行して、また立て直す」という繰り返しが典型的です。  
51 回（365 日 / 7 日スライド）モデルを再構築・再求解します。

```
day:  0  1  2  3  4  5  6  7  8   ← 9 日ウィンドウ
      [=====実行=====][look-ahead]
                  ↓ 7 日スライド
      7  8  9 10 11 12 13 14 15
```

**PuLP**: ウィンドウごとに全変数・全制約を作り直す  
**linopy**: 1 回構築したモデルの **RHS だけを更新** して再求解

In [ ]:
# 問題パラメータ
N_PRODUCTS = 500
N_DAYS     = 365
WINDOW     = 9
SLIDE      = 7

RNG = np.random.default_rng(42)
demand         = RNG.integers(10, 100, size=(N_PRODUCTS, N_DAYS)).astype(float)
max_inventory  = RNG.integers(200, 500, size=N_PRODUCTS).astype(float)
initial_inventory = (max_inventory * 0.3).astype(float)
production_cost   = RNG.uniform(1.0, 5.0, size=N_PRODUCTS)
inventory_cost    = RNG.uniform(0.1, 0.5, size=N_PRODUCTS)

n_windows = len(range(0, N_DAYS - WINDOW + 1, SLIDE))
print(f"{N_PRODUCTS} 製品 × {N_DAYS} 日 → {n_windows} ウィンドウ")

### PuLP 実装

毎ウィンドウ `LpProblem` を作り直します。

In [ ]:
def solve_window_pulp(start, inventory_init):
    end    = min(start + WINDOW, N_DAYS)
    n_days = end - start
    days, products = list(range(n_days)), list(range(N_PRODUCTS))

    t0 = time.perf_counter()
    prob = pulp.LpProblem("prod", pulp.LpMinimize)
    x = {(p, t): pulp.LpVariable(f"x_{p}_{t}", lowBound=0)
         for p in products for t in days}
    s = {(p, t): pulp.LpVariable(f"s_{p}_{t}", lowBound=0, upBound=max_inventory[p])
         for p in products for t in days}
    prob += pulp.LpAffineExpression(
        [(x[p, t], production_cost[p]) for p in products for t in days]
        + [(s[p, t], inventory_cost[p]) for p in products for t in days]
    )
    for p in products:
        for t in days:
            prev = inventory_init[p] if t == 0 else s[p, t - 1]
            prob += s[p, t] == prev + x[p, t] - demand[p, start + t]
    t_build = time.perf_counter() - t0

    prob.solve(pulp.HiGHS(msg=False))
    t_solve = time.perf_counter() - t0 - t_build

    inv_end = np.array([pulp.value(s[p, SLIDE - 1]) for p in products])
    return pulp.value(prob.objective), inv_end, t_build, t_solve


def run_pulp():
    inventory = initial_inventory.copy()
    results = {"objectives": [], "build_times": [], "solve_times": []}
    for start in range(0, N_DAYS - WINDOW + 1, SLIDE):
        obj, inventory, bt, st = solve_window_pulp(start, inventory)
        results["objectives"].append(obj)
        results["build_times"].append(bt)
        results["solve_times"].append(st)
    return results

### linopy 実装

linopy の強みは **モデルを 1 回だけ構築し、RHS 更新だけで再求解** できること。  
また `add_constraints` を day 次元ごとではなく **1 回のみ呼ぶ** ことで構築も高速です。

In [ ]:
_products_l  = np.arange(N_PRODUCTS)
_days_l      = np.arange(WINDOW)
_inner_days  = np.arange(1, WINDOW)
_da_max_inv  = xr.DataArray(max_inventory, dims=["product"])
_da_prod_cost = xr.DataArray(production_cost, dims=["product"])
_da_inv_cost  = xr.DataArray(inventory_cost,  dims=["product"])

SOLVER_OPTS = {
    "solver_name": "highs",
    "io_api": "direct",
    "output_flag": False,
    "log_to_console": False,
}


def build_model(inventory_init, start):
    """モデルを 1 回だけ構築する。"""
    m = linopy.Model()
    coords = {"product": _products_l, "day": _days_l}
    x = m.add_variables(lower=0, coords=coords, name="x")
    s = m.add_variables(lower=0, upper=_da_max_inv, coords=coords, name="s")

    da_d    = xr.DataArray(demand[:, start:start + WINDOW], dims=["product", "day"])
    da_init = xr.DataArray(inventory_init, dims=["product"])

    # day=0 制約（1 本）
    m.add_constraints(
        s.isel(day=0) - x.isel(day=0) == da_init - da_d.isel(day=0),
        name="balance_0",
    )
    # day=1..WINDOW-1 制約（day 次元を一括追加 → 1 回の呼び出し）
    s_cur  = s.isel(day=slice(1, None)).assign_coords(day=_inner_days)
    s_prev = s.isel(day=slice(0, -1)).assign_coords(day=_inner_days)
    x_cur  = x.isel(day=slice(1, None)).assign_coords(day=_inner_days)
    m.add_constraints(
        s_cur - s_prev - x_cur
        == -da_d.isel(day=slice(1, None)).assign_coords(day=_inner_days),
        name="balance_inner",
    )
    m.add_objective((_da_prod_cost * x + _da_inv_cost * s).sum())
    return m


def update_model(m, inventory_init, start):
    """需要データと初期在庫が変わった分だけ RHS を書き換える。"""
    da_d    = xr.DataArray(demand[:, start:start + WINDOW], dims=["product", "day"])
    da_init = xr.DataArray(inventory_init, dims=["product"])
    m.constraints["balance_0"].rhs     = (da_init - da_d.isel(day=0)).values
    m.constraints["balance_inner"].rhs = (-da_d.isel(day=slice(1, None))).values


def run_linopy():
    inventory = initial_inventory.copy()
    results = {"objectives": [], "build_times": [], "solve_times": []}

    t0 = time.perf_counter()
    m = build_model(inventory, start=0)
    initial_build = time.perf_counter() - t0

    for i, start in enumerate(range(0, N_DAYS - WINDOW + 1, SLIDE)):
        t0 = time.perf_counter()
        if i > 0:
            update_model(m, inventory, start)
        t_build = time.perf_counter() - t0

        m.solve(**SOLVER_OPTS, progress=False)
        t_solve = time.perf_counter() - t0 - t_build

        inventory = np.maximum(m.solution["s"].isel(day=SLIDE - 1).values, 0)
        results["objectives"].append(float(m.objective.value))
        results["build_times"].append(initial_build if i == 0 else t_build)
        results["solve_times"].append(t_solve)

    return results

## 4. ベンチマーク実行

両方を実行して結果を比較します。数分かかります。

In [ ]:
print("PuLP (HiGHS) を実行中 ...")
t0 = time.perf_counter()
res_pulp = run_pulp()
t_pulp = time.perf_counter() - t0
print(f"  完了: {t_pulp:.2f}s")

print("linopy (HiGHS direct) を実行中 ...")
t0 = time.perf_counter()
res_linopy = run_linopy()
t_linopy = time.perf_counter() - t0
print(f"  完了: {t_linopy:.2f}s")

In [ ]:
bt_p = np.array(res_pulp["build_times"])
st_p = np.array(res_pulp["solve_times"])
bt_l = np.array(res_linopy["build_times"])
st_l = np.array(res_linopy["solve_times"])

sep = "-" * 62
print(f"{'':30s} {'PuLP':>10s} {'linopy':>10s} {'speedup':>10s}")
print(sep)
print(f"{'total wall time (s)':30s} {t_pulp:>10.2f} {t_linopy:>10.2f} {t_pulp/t_linopy:>9.1f}x")
print(f"{'build/update avg (ms)':30s} {np.mean(bt_p)*1e3:>10.1f} {np.mean(bt_l)*1e3:>10.1f} {np.mean(bt_p)/np.mean(bt_l):>9.1f}x")
print(f"{'  first build (ms)':30s} {bt_p[0]*1e3:>10.1f} {bt_l[0]*1e3:>10.1f} {bt_p[0]/bt_l[0]:>9.1f}x")
print(f"{'  update avg (ms)':30s} {np.mean(bt_p[1:])*1e3:>10.1f} {np.mean(bt_l[1:])*1e3:>10.1f} {np.mean(bt_p[1:])/np.mean(bt_l[1:]):>9.1f}x")
print(f"{'solve avg (ms)':30s} {np.mean(st_p)*1e3:>10.1f} {np.mean(st_l)*1e3:>10.1f} {np.mean(st_p)/np.mean(st_l):>9.1f}x")
print(f"{'cumulative build time (s)':30s} {bt_p.sum():>10.2f} {bt_l.sum():>10.2f} {bt_p.sum()/bt_l.sum():>9.1f}x")

## 5. メモリ効率

`tracemalloc` でモデル構築時のピークメモリを計測します。

PuLP は変数・制約を Python オブジェクトとして個別に保持するのに対し、  
linopy は numpy/xarray 配列に格納するため、**メモリ効率が大きく異なります**。

In [ ]:
def measure_peak_mb(func):
    gc.collect()
    tracemalloc.start()
    func()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024 / 1024


def _build_pulp_once():
    products, days = list(range(N_PRODUCTS)), list(range(WINDOW))
    prob = pulp.LpProblem("mem_test", pulp.LpMinimize)
    x = {(p, t): pulp.LpVariable(f"x_{p}_{t}", lowBound=0)
         for p in products for t in days}
    s = {(p, t): pulp.LpVariable(f"s_{p}_{t}", lowBound=0, upBound=max_inventory[p])
         for p in products for t in days}
    prob += pulp.LpAffineExpression(
        [(x[p, t], production_cost[p]) for p in products for t in days]
        + [(s[p, t], inventory_cost[p]) for p in products for t in days]
    )
    for p in products:
        for t in days:
            prev = initial_inventory[p] if t == 0 else s[p, t - 1]
            prob += s[p, t] == prev + x[p, t] - demand[p, t]


def _build_linopy_once():
    coords = {"product": np.arange(N_PRODUCTS), "day": np.arange(WINDOW)}
    da_d    = xr.DataArray(demand[:, :WINDOW], dims=["product", "day"])
    da_init = xr.DataArray(initial_inventory, dims=["product"])
    m = linopy.Model()
    x = m.add_variables(lower=0, coords=coords, name="x")
    s = m.add_variables(lower=0, upper=_da_max_inv, coords=coords, name="s")
    m.add_constraints(
        s.isel(day=0) - x.isel(day=0) == da_init - da_d.isel(day=0),
        name="balance_0",
    )
    s_cur  = s.isel(day=slice(1, None)).assign_coords(day=_inner_days)
    s_prev = s.isel(day=slice(0, -1)).assign_coords(day=_inner_days)
    x_cur  = x.isel(day=slice(1, None)).assign_coords(day=_inner_days)
    m.add_constraints(
        s_cur - s_prev - x_cur
        == -da_d.isel(day=slice(1, None)).assign_coords(day=_inner_days),
        name="balance_inner",
    )
    m.add_objective((_da_prod_cost * x + _da_inv_cost * s).sum())


N_TRIALS = 5
pulp_peaks   = [measure_peak_mb(_build_pulp_once)   for _ in range(N_TRIALS)]
linopy_peaks = [measure_peak_mb(_build_linopy_once) for _ in range(N_TRIALS)]

p_med = np.median(pulp_peaks)
l_med = np.median(linopy_peaks)

print(f"=== Memory Benchmark (N_PRODUCTS={N_PRODUCTS}, WINDOW={WINDOW}) ===")
print(f"{'':30s} {'peak MB':>10s}")
print("-" * 42)
print(f"{'PuLP (model build)':30s} {p_med:>10.1f}")
print(f"{'linopy (model build)':30s} {l_med:>10.1f}")
print(f"{'ratio':30s} {p_med/l_med:>10.1f}x")

## まとめ

| 特長 | PuLP | linopy |
|------|------|--------|
| 変数・制約の定義 | Python 辞書 + ループ | xarray 配列（ループ不要） |
| モデル更新 | 毎回再構築 | RHS だけ更新 |
| 初回ビルド速度 | baseline | **2.2x 高速** |
| update 速度 | baseline | **3.8x 高速** |
| solve 速度 | baseline | **1.3x 高速** |
| ピークメモリ | baseline | **7.5x 少ない** |
| 合計実行時間 | baseline | **1.9x 高速** |

linopy の性能を引き出すには **2 つのコツ** があります：

1. `add_constraints` を for ループで呼ばない → xarray で次元を揃えて **1 回で渡す**
2. ローリングホライゾンでは `update_model` で **RHS だけ書き換える**（モデル再構築なし）

---

**参考リンク**
- [linopy GitHub](https://github.com/PyPSA/linopy)
- [linopy JOSS paper](https://joss.theoj.org/papers/10.21105/joss.04823)
- [linopy Issue #198: Iterative HiGHS solve](https://github.com/PyPSA/linopy/issues/198)